# Dataset Explorer

Selecciona uno o varios archivos `.xdf` del dataset y revisa canales etiquetados, secuencia de markers y orden de estimulos por trial.

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime
from html import escape

import pyxdf
from IPython.display import HTML, display

DATASET_DIR = Path('../dataset')


def display_table(rows, columns=None):
    rows = list(rows)
    if not rows:
        display(HTML('<p><em>Sin datos.</em></p>'))
        return

    columns = columns or list(rows[0].keys())
    html = ['<table>']
    html.append('<thead><tr>' + ''.join(f'<th>{escape(str(column))}</th>' for column in columns) + '</tr></thead>')
    html.append('<tbody>')
    for row in rows:
        html.append('<tr>' + ''.join(f'<td>{escape(str(row.get(column, "")))}</td>' for column in columns) + '</tr>')
    html.append('</tbody></table>')
    display(HTML(''.join(html)))


xdf_files = sorted(
    (path for path in DATASET_DIR.rglob('*.xdf') if path.is_file()),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not xdf_files:
    raise FileNotFoundError(f'No se encontraron archivos .xdf en {DATASET_DIR.resolve()}')

file_rows = []
for index, path in enumerate(xdf_files):
    stat = path.stat()
    file_rows.append({
        'index': index,
        'archivo': str(path.relative_to(DATASET_DIR)),
        'modificado': datetime.fromtimestamp(stat.st_mtime).strftime('%Y-%m-%d %H:%M:%S'),
        'tamano_MB': round(stat.st_size / (1024 * 1024), 2),
    })

display_table(file_rows, ['index', 'archivo', 'modificado', 'tamano_MB'])

# Opcion A: si tienes ipywidgets, selecciona desde el widget y luego ejecuta la siguiente celda.
try:
    import ipywidgets as widgets

    xdf_options = [str(path.relative_to(DATASET_DIR)) for path in xdf_files]
    xdf_selector = widgets.SelectMultiple(
        options=xdf_options,
        value=(xdf_options[0],),
        rows=min(12, len(xdf_options)),
        description='XDF',
        layout=widgets.Layout(width='95%'),
    )
    display(xdf_selector)
except Exception as error:
    xdf_selector = None
    print('Selector interactivo no disponible; usa selected_xdf_indices en la siguiente celda.')
    print(error)


In [ ]:
# Opcion B: seleccion manual por indices de la tabla anterior.
# Puedes poner uno o varios indices, por ejemplo: [0] o [0, 2, 5]
selected_xdf_indices = [0]

if xdf_selector is not None and tuple(xdf_selector.value):
    selected_xdf_paths = [DATASET_DIR / value for value in xdf_selector.value]
else:
    selected_xdf_paths = [xdf_files[index] for index in selected_xdf_indices]

print('Archivos seleccionados:')
for index, path in enumerate(selected_xdf_paths, start=1):
    print(f'{index}. {path}')


In [ ]:
STANDARD_UNICORN_CHANNEL_LABELS = ['Fz', 'C3', 'Cz', 'C4', 'Pz', 'PO7', 'Oz', 'PO8']
TASK_PREFIXES = ('MI_', 'MO_', 'AW_', 'ME_')
CLASS_PREFIX = 'CLASS_'

CLASS_LABELS = {
    'ARM': 'ARMS',
    'LEG': 'LEGS',
    'LEFT': 'LEFT',
    'RIGHT': 'RIGHT',
}


def first_value(value, default=''):
    if isinstance(value, list):
        return first_value(value[0], default) if value else default
    if isinstance(value, tuple):
        return first_value(value[0], default) if value else default
    if value is None:
        return default
    return value


def marker_value(sample):
    if isinstance(sample, (list, tuple)):
        return str(first_value(sample, ''))
    return str(sample)


def stream_name(stream):
    return str(first_value(stream.get('info', {}).get('name'), 'unknown'))


def stream_type(stream):
    return str(first_value(stream.get('info', {}).get('type'), 'unknown'))


def stream_channel_count(stream):
    try:
        return int(first_value(stream.get('info', {}).get('channel_count'), 0) or 0)
    except ValueError:
        return 0


def extract_xdf_channel_labels(info):
    labels = []
    try:
        desc = first_value(info.get('desc'), {})
        channels_root = first_value(desc.get('channels'), {}) if isinstance(desc, dict) else {}
        channel_nodes = channels_root.get('channel', []) if isinstance(channels_root, dict) else []
        if isinstance(channel_nodes, dict):
            channel_nodes = [channel_nodes]
        for channel in channel_nodes:
            if not isinstance(channel, dict):
                continue
            label = first_value(channel.get('label')) or first_value(channel.get('name'))
            if label:
                labels.append(str(label))
    except Exception:
        pass
    return labels


def fallback_channel_labels(channel_count):
    if channel_count <= len(STANDARD_UNICORN_CHANNEL_LABELS):
        return STANDARD_UNICORN_CHANNEL_LABELS[:channel_count]
    return [f'EEG{index}' for index in range(1, channel_count + 1)]


def select_marker_stream(streams):
    for stream in streams:
        if stream_type(stream).lower() == 'markers':
            return stream
    for stream in streams:
        name = stream_name(stream).lower()
        if 'marker' in name:
            return stream
    return None


def select_eeg_stream(streams):
    candidates = []
    for stream in streams:
        name = stream_name(stream).lower()
        kind = stream_type(stream).lower()
        count = stream_channel_count(stream)
        priority = 0 if kind == 'eeg' else 1 if 'eeg' in name else 2
        candidates.append((priority, abs(count - 8), -count, stream))
    if not candidates:
        return None
    return sorted(candidates, key=lambda item: item[:3])[0][3]


def load_xdf(path):
    streams, header = pyxdf.load_xdf(str(path))
    return streams, header


def extract_markers(streams):
    marker_stream = select_marker_stream(streams)
    if marker_stream is None:
        return []
    return [marker_value(sample) for sample in marker_stream.get('time_series', [])]


def parse_stimulus_marker(marker):
    for prefix in TASK_PREFIXES:
        if marker.startswith(prefix):
            modality = prefix.rstrip('_')
            payload = marker[len(prefix):]
            parts = payload.split('_')
            class_key = parts[0] if parts else payload
            class_label = CLASS_LABELS.get(class_key, class_key)
            return {
                'modalidad': modality,
                'clase': class_label,
                'marker': marker,
                'estimulo': f'{modality}_{class_label}',
            }
    return None


def extract_stimulus_order(markers):
    rows = []
    current_trial = None
    current_class = None

    for marker in markers:
        if marker.startswith('TRIAL_'):
            try:
                current_trial = int(marker.split('_', 1)[1])
            except ValueError:
                current_trial = marker
            current_class = None
            continue

        if marker.startswith(CLASS_PREFIX):
            raw_class = marker[len(CLASS_PREFIX):]
            current_class = CLASS_LABELS.get(raw_class, raw_class)
            continue

        stimulus = parse_stimulus_marker(marker)
        if stimulus:
            rows.append({
                'orden': len(rows) + 1,
                'trial': current_trial,
                'estimulo': stimulus['estimulo'],
                'modalidad': stimulus['modalidad'],
                'clase_marker': stimulus['clase'],
                'clase_trial': current_class,
                'marker': stimulus['marker'],
            })

    return rows


In [ ]:
analysis_results = []

for xdf_path in selected_xdf_paths:
    streams, header = load_xdf(xdf_path)
    eeg_stream = select_eeg_stream(streams)
    markers = extract_markers(streams)
    stimulus_order = extract_stimulus_order(markers)

    if eeg_stream is None:
        channel_count = 0
        tagged_channel_labels = []
        display_channel_labels = []
        eeg_stream_info = {'name': 'no encontrado', 'type': 'no encontrado'}
    else:
        info = eeg_stream.get('info', {})
        channel_count = stream_channel_count(eeg_stream)
        tagged_channel_labels = extract_xdf_channel_labels(info)
        display_channel_labels = tagged_channel_labels or fallback_channel_labels(channel_count)
        eeg_stream_info = {'name': stream_name(eeg_stream), 'type': stream_type(eeg_stream)}

    analysis_results.append({
        'path': xdf_path,
        'streams': streams,
        'eeg_stream': eeg_stream,
        'eeg_stream_info': eeg_stream_info,
        'channel_count': channel_count,
        'tagged_channel_labels': tagged_channel_labels,
        'display_channel_labels': display_channel_labels,
        'markers': markers,
        'stimulus_order': stimulus_order,
    })

print(f'Archivos cargados: {len(analysis_results)}')


## Canales etiquetados

In [ ]:
channel_rows = []

for file_index, result in enumerate(analysis_results, start=1):
    labels = result['tagged_channel_labels']
    fallback = result['display_channel_labels']
    channel_rows.append({
        'archivo': file_index,
        'ruta': str(result['path']),
        'stream_eeg': result['eeg_stream_info']['name'],
        'tipo_stream': result['eeg_stream_info']['type'],
        'cantidad_canales': result['channel_count'],
        'canales_etiquetados': ', '.join(labels) if labels else 'sin etiquetas en XDF',
        'canales_mostrados': ', '.join(fallback) if fallback else 'sin canales detectados',
    })

display_table(channel_rows, [
    'archivo',
    'ruta',
    'stream_eeg',
    'tipo_stream',
    'cantidad_canales',
    'canales_etiquetados',
    'canales_mostrados',
])


## Secuencia de markers

In [ ]:
for file_index, result in enumerate(analysis_results, start=1):
    print('=' * 100)
    print(f'Archivo {file_index}: {result["path"]}')
    print(f'Total markers: {len(result["markers"])}')
    print()
    for index, marker in enumerate(result['markers'], start=1):
        print(f'{index:03d}. {marker}')


## Orden de estimulos

In [ ]:
for file_index, result in enumerate(analysis_results, start=1):
    print('=' * 100)
    print(f'Archivo {file_index}: {result["path"]}')
    stimulus_order = result['stimulus_order']
    if not stimulus_order:
        print('No se encontraron markers de estimulo tipo MI_, MO_, AW_ o ME_.')
        continue

    display_table(stimulus_order, ['orden', 'trial', 'estimulo', 'modalidad', 'clase_marker', 'clase_trial', 'marker'])
    print('Resumen por estimulo:')
    counts = Counter(row['estimulo'] for row in stimulus_order)
    display_table(
        [{'estimulo': key, 'conteo': value} for key, value in sorted(counts.items())],
        ['estimulo', 'conteo'],
    )


## Resumen rapido

In [ ]:
summary_rows = []

for file_index, result in enumerate(analysis_results, start=1):
    stimulus_order = result['stimulus_order']
    modality_counts = Counter(row['modalidad'] for row in stimulus_order)
    summary_rows.append({
        'archivo': file_index,
        'ruta': str(result['path']),
        'markers': len(result['markers']),
        'estimulos': len(stimulus_order),
        'MI': modality_counts.get('MI', 0),
        'MO': modality_counts.get('MO', 0),
        'AW': modality_counts.get('AW', 0),
        'ME': modality_counts.get('ME', 0),
    })

display_table(summary_rows, ['archivo', 'ruta', 'markers', 'estimulos', 'MI', 'MO', 'AW', 'ME'])
